In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification

In [ ]:
# ==========================================
# 1. LOAD AND PREPARE DATA
# ==========================================
df = pd.read_csv('products_sample.csv')

# Combine Name and Description for better context
df['text'] = df['product_name'].astype(str) + " " + df['product_desc'].astype(str)

# Encode category names into numbers
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['cat_name'])
num_labels = len(label_encoder.classes_)

# Split into Train and Validation sets (80/20)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

In [ ]:
# ==========================================
# 2. TOKENIZATION
# ==========================================
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_data(texts):
    return tokenizer(
        texts, 
        padding=True, 
        truncation=True, 
        max_length=256, 
        return_tensors="tf"
    )

train_encodings = tokenize_data(train_texts)
val_encodings = tokenize_data(val_texts)

# Convert to TF Datasets for better performance
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).shuffle(len(train_texts)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(64)

In [ ]:
# ==========================================
# 3. MODEL SETUP (Includes SafeTensors Fix)
# ==========================================
model = TFDistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=num_labels,
    use_safetensors=False  # Crucial fix for the error you encountered
)

optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy('accuracy')]

model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

In [ ]:
# 4. TRAINING
# ==========================================
print("Starting training...")
# Define the callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=1,          # Stop if it doesn't improve for 1 epoch
    restore_best_weights=True
)

# Train with the callback
model.fit(
    train_dataset, 
    validation_data=val_dataset, 
    epochs=10,           # Set a high max, but it will likely stop at 3 or 4
    callbacks=[early_stopping]
)

In [ ]:
# ==========================================
# 5. MULTI-CATEGORY ACCURACY REPORT
# ==========================================
print("\nEvaluating Model...")
# Get raw predictions
predictions = model.predict(val_dataset)
# Convert logits to class index
y_pred = np.argmax(predictions.logits, axis=1)

# Extract true labels from the val_dataset
y_true = np.concatenate([y for x, y in val_dataset], axis=0)

# Generate detailed report
report = classification_report(
    y_true, 
    y_pred, 
    target_names=label_encoder.classes_
)

print("\n--- Detailed Category Report ---")
print(report)
print(f"Overall Accuracy: {accuracy_score(y_true, y_pred):.2%}")

In [ ]:
def predict_top_categories(name, desc, k=3):
    text = f"{name} {desc}"
    inputs = tokenizer(text, return_tensors="tf", truncation=True, padding=True, max_length=128)
    
    outputs = model(inputs)[0]
    # Use Softmax because the model was trained for single-label
    probs = tf.nn.softmax(outputs, axis=-1).numpy()[0]
    
    # Get indices of the top K highest probabilities
    top_indices = np.argsort(probs)[-k:][::-1]
    
    results = []
    for idx in top_indices:
        category = label_encoder.inverse_transform([idx])[0]
        score = probs[idx]
        results.append((category, score))
    
    return results

# Example output: [('Electronics', 0.85), ('Gadgets', 0.12), ('Home Office', 0.02)]

In [ ]:
# 1. Define your product data
product_name = "YATTA Golf® - Summer Ice women's Moisture Wicking non iron Golf Polo Shirts"
product_desc = "Made of 95% polyester and 5% spandex 4-way stretch for maximum mobility Moisture-wicking fabric keeps you dry Wrinkle-resistant for a polished look Athletic fit enhances performance Easy care: machine wash cold, tumble dry low"

# 2. Call the function (k=3 means get the top 3 suggestions)
results = predict_top_categories(product_name, product_desc, k=3)

# 3. Loop through and print the results
print(f"\n--- Predictions for: {product_name[:50]}... ---")

for category, confidence in results:
    print(f"Category: {category:<20} | Confidence: {confidence:.2%}")

In [ ]:
import pickle
import os

# Create a folder for the model if it doesn't exist
model_directory = "./product_model"
if not os.path.exists(model_directory):
    os.makedirs(model_directory)

# 1. Save the TensorFlow Model and Tokenizer
# This saves 'config.json', 'tf_model.h5', 'vocab.txt', etc.
model.save_pretrained(model_directory)
tokenizer.save_pretrained(model_directory)

# 2. Save the Label Encoder
# This is crucial so your app knows 0='Electronics', 1='Clothing', etc.
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print(f"✅ All assets saved to {model_directory} and label_encoder.pkl")